# Week 7 Maintainer's Copilot — LLM Baseline

This notebook runs the third required classifier:

```text
LLM baseline
```

It uses the same validated dataset as the classical and transformer models:

```text
issues_processed_pandas_label_fetch.csv
```

Task:

Classify each test issue into exactly one label:

- `bug`
- `feature`
- `docs`
- `question`

Outputs:

- `llm_baseline_predictions_pandas.csv`
- `llm_baseline_metrics_pandas.json`
- `llm_baseline_confusion_matrix_pandas.png`

Start with a small sample first, then run the full test set only when everything works.


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib tqdm openai

In [ ]:
import os
import re
import json
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from openai import OpenAI


In [ ]:
DATA_PATH = Path("issues_processed_pandas_label_fetch.csv")

PROJECT_LABELS = ["bug", "feature", "docs", "question"]

# Low-cost LLM baseline model. Change if your provider/instructor requires another model.
LLM_MODEL = "gpt-4o-mini"

# Start with sample. After it works, change to "full".
RUN_MODE = "sample"  # "sample" or "full"
SAMPLE_SIZE = 20

MAX_CHARS_PER_ISSUE = 3500

PREDICTIONS_PATH = Path("llm_baseline_predictions_pandas.csv")
METRICS_PATH = Path("llm_baseline_metrics_pandas.json")
CONFUSION_MATRIX_PATH = Path("llm_baseline_confusion_matrix_pandas.png")

# Approximate pricing variables for gpt-4o-mini.
# Update these if you use another model.
INPUT_COST_PER_1M_TOKENS = 0.15
OUTPUT_COST_PER_1M_TOKENS = 0.60


## API key

Run this in Colab and paste your OpenAI API key when prompted.

The key is stored only in the Colab runtime environment.


In [ ]:
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("API key configured:", bool(os.getenv("OPENAI_API_KEY")))


## Load the test split

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} was not found. Upload issues_processed_pandas_label_fetch.csv first."
    )

df = pd.read_csv(DATA_PATH)

required_columns = {"clean_text", "mapped_label", "split", "title", "url"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

test_df = df[df["split"] == "test"].copy().reset_index(drop=True)

print("Full test size:", len(test_df))
display(test_df["mapped_label"].value_counts())

if RUN_MODE == "sample":
    eval_df = test_df.sample(n=min(SAMPLE_SIZE, len(test_df)), random_state=42).reset_index(drop=True)
else:
    eval_df = test_df.copy()

print("Evaluation size:", len(eval_df))
display(eval_df["mapped_label"].value_counts())


## Prompt

The LLM must return JSON only so the output can be evaluated automatically.


In [ ]:
SYSTEM_PROMPT = """You are an expert open-source maintainer triaging GitHub issues.

Classify the issue into exactly one of these labels:
- bug: broken behavior, error, regression, incorrect result, crash, failing behavior
- feature: request for a new capability, enhancement, API addition, or behavior improvement
- docs: documentation problem, missing docs, unclear docs, examples, tutorials
- question: usage question, help request, clarification, how-to question

Return only valid JSON with this schema:
{
  "label": "bug | feature | docs | question",
  "reason": "short reason"
}

Do not include markdown.
"""


def build_user_prompt(title: str, text: str) -> str:
    issue_text = str(text or "")
    issue_text = issue_text[:MAX_CHARS_PER_ISSUE]

    return f"""GitHub issue title:
{title}

GitHub issue text:
{issue_text}

Classify this issue into exactly one label: bug, feature, docs, or question.
"""


## LLM call helper

In [ ]:
def extract_json_object(text: str) -> dict:
    """Parse JSON from the model response, with a fallback for accidental extra text."""
    text = (text or "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        return json.loads(match.group(0))

    raise ValueError(f"Could not parse JSON from response: {text[:300]}")


def normalize_label(label: str) -> str:
    """Normalize model label into one of the project labels."""
    label = str(label or "").strip().lower()

    if label in PROJECT_LABELS:
        return label

    if "bug" in label:
        return "bug"

    if "feature" in label or "enhancement" in label:
        return "feature"

    if "doc" in label:
        return "docs"

    if "question" in label or "usage" in label or "help" in label:
        return "question"

    return "unknown"


def classify_with_llm(row: pd.Series) -> dict:
    """Call the LLM once for one issue and return prediction metadata."""
    title = row.get("title", "")
    text = row.get("clean_text", "")

    user_prompt = build_user_prompt(title, text)

    start_time = time.perf_counter()

    response = client.responses.create(
        model=LLM_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_output_tokens=120,
    )

    latency_seconds = time.perf_counter() - start_time

    output_text = response.output_text
    parsed = extract_json_object(output_text)

    predicted_label = normalize_label(parsed.get("label"))
    reason = str(parsed.get("reason", ""))

    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", 0) if usage else 0
    output_tokens = getattr(usage, "output_tokens", 0) if usage else 0

    estimated_cost = (
        (input_tokens / 1_000_000) * INPUT_COST_PER_1M_TOKENS
        + (output_tokens / 1_000_000) * OUTPUT_COST_PER_1M_TOKENS
    )

    return {
        "predicted_label": predicted_label,
        "reason": reason,
        "raw_response": output_text,
        "latency_seconds": latency_seconds,
        "input_tokens": int(input_tokens or 0),
        "output_tokens": int(output_tokens or 0),
        "estimated_cost_usd": float(estimated_cost),
    }


## Run LLM baseline

This cell may cost money because it calls the API.

Start with:

```python
RUN_MODE = "sample"
```

After the sample works, change to:

```python
RUN_MODE = "full"
```

and rerun from the load-test-split cell onward.


In [ ]:
results = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    try:
        prediction = classify_with_llm(row)
        status = "ok"
        error = ""
    except Exception as exc:
        prediction = {
            "predicted_label": "unknown",
            "reason": "",
            "raw_response": "",
            "latency_seconds": 0.0,
            "input_tokens": 0,
            "output_tokens": 0,
            "estimated_cost_usd": 0.0,
        }
        status = "error"
        error = str(exc)

    results.append({
        "github_issue_id": row.get("github_issue_id"),
        "url": row.get("url"),
        "title": row.get("title"),
        "true_label": row.get("mapped_label"),
        "predicted_label": prediction["predicted_label"],
        "reason": prediction["reason"],
        "status": status,
        "error": error,
        "latency_seconds": prediction["latency_seconds"],
        "input_tokens": prediction["input_tokens"],
        "output_tokens": prediction["output_tokens"],
        "estimated_cost_usd": prediction["estimated_cost_usd"],
        "raw_response": prediction["raw_response"],
    })

    time.sleep(0.2)

pred_df = pd.DataFrame(results)
pred_df.to_csv(PREDICTIONS_PATH, index=False)

print(f"Saved predictions to: {PREDICTIONS_PATH}")
display(pred_df.head())
print(pred_df["status"].value_counts())


## Evaluate LLM baseline

In [ ]:
valid_pred_df = pred_df[pred_df["predicted_label"].isin(PROJECT_LABELS)].copy()

if len(valid_pred_df) == 0:
    raise ValueError("No valid LLM predictions to evaluate.")

y_true = valid_pred_df["true_label"].tolist()
y_pred = valid_pred_df["predicted_label"].tolist()

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")

print("LLM baseline accuracy:", accuracy)
print("LLM baseline macro-F1:", macro_f1)
print("LLM baseline weighted-F1:", weighted_f1)

print("\nClassification report:")
print(classification_report(y_true, y_pred, labels=PROJECT_LABELS, zero_division=0))


## Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=PROJECT_LABELS)

plt.figure(figsize=(7, 5))
plt.imshow(cm)
plt.title("LLM Baseline Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(range(len(PROJECT_LABELS)), PROJECT_LABELS, rotation=45)
plt.yticks(range(len(PROJECT_LABELS)), PROJECT_LABELS)
plt.colorbar()

for i in range(len(PROJECT_LABELS)):
    for j in range(len(PROJECT_LABELS)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=150)
plt.show()

print(f"Saved confusion matrix to: {CONFUSION_MATRIX_PATH}")


## Save metrics

In [ ]:
per_class_report = classification_report(
    y_true,
    y_pred,
    labels=PROJECT_LABELS,
    output_dict=True,
    zero_division=0,
)

metrics = {
    "repo": "pandas-dev/pandas",
    "model": LLM_MODEL,
    "task": "GitHub issue classification",
    "run_mode": RUN_MODE,
    "num_examples": int(len(valid_pred_df)),
    "labels": PROJECT_LABELS,
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "per_class_report": per_class_report,
    "latency": {
        "average_seconds": float(valid_pred_df["latency_seconds"].mean()),
        "median_seconds": float(valid_pred_df["latency_seconds"].median()),
        "total_seconds": float(valid_pred_df["latency_seconds"].sum()),
    },
    "tokens": {
        "total_input_tokens": int(valid_pred_df["input_tokens"].sum()),
        "total_output_tokens": int(valid_pred_df["output_tokens"].sum()),
    },
    "cost": {
        "estimated_total_usd": float(valid_pred_df["estimated_cost_usd"].sum()),
        "input_cost_per_1m_tokens": INPUT_COST_PER_1M_TOKENS,
        "output_cost_per_1m_tokens": OUTPUT_COST_PER_1M_TOKENS,
    },
    "created_at": datetime.utcnow().isoformat() + "Z",
}

with METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"Saved metrics to: {METRICS_PATH}")
metrics


# Send back

After the sample works and after running full mode, send:

```text
llm_baseline_metrics_pandas.json
llm_baseline_confusion_matrix_pandas.png
llm_baseline_predictions_pandas.csv
```
